# Train the Student — Gemma-3-270M on Hop-1 Decomposition

Notebook 02 labeled HotpotQA with GPT-4o and formatted the rows into chat-template `messages`. Notebook 03 built the scoring harness. Here I fine-tune Gemma-3-270M on the teacher's Hop-1 decompositions and generate first-step decompositions for the 95 held-out test questions.

This notebook trains one ablation variant at a time. It runs on a Colab GPU (T4 or L4). The output is `preds_{variant}.jsonl`, which I download and score locally through notebook 03.

Variant this run: **sys** (a short fixed `system` turn prepended to `user`/`assistant`).

## 1. Environment and GPU

In [1]:
!pip install -q -U transformers trl datasets accelerate huggingface_hub

import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

CUDA: True
GPU : Tesla T4


Gemma-3-270M is a gated model on the Hub, so I authenticate with a read token (accept the license on the model page first). The token lives only in this session.

In [2]:
import getpass
from huggingface_hub import login

login(getpass.getpass("Hugging Face token: "))

Hugging Face token: ··········


## 2. Load the training data

I upload the two sys files from `data/`: `train_sys.jsonl` and `test_sys.jsonl`. Training only needs the `messages` column; the extra fields (`gold_titles`, `type`, and so on) are kept in the test file for scoring later.

In [ ]:
from google.colab import files

# Upload train_sys.jsonl and test_sys.jsonl when prompted.
files.upload()

In [1]:
from datasets import load_dataset

VARIANT = "sys"

# SFTTrainer reads the conversational `messages` column and applies the chat
# template itself, so I drop every other column from the training set.
train_ds = load_dataset(
    "json", data_files=f"train_{VARIANT}.jsonl", split="train"
).select_columns(["messages"])

print(train_ds)
print(train_ds[0]["messages"])

FileNotFoundError: Unable to find '/Users/mac/Projects/hop-specialist/notebooks/train_sys.jsonl'

## 3. Load the base model and tokenizer

In [5]:
# from transformers import pipeline

# pipe = pipeline("text-generation", model="google/gemma-3-270m-it")
# message = [
#     {"role": "user", "content": "Who are you?"},
# ]
# pipe(message)

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-3-270m-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Pin fp32 explicitly: recent transformers defaults from_pretrained to the
# checkpoint's own dtype, and Gemma-3's checkpoint is bf16. T4 has no bf16
# tensor cores, so that silent default broke the fp16 GradScaler below.
# fp32 weights + fp16=True in SFTConfig is the standard T4 mixed-precision
# setup: master weights stay fp32, autocast handles the fp16 compute.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

print(model.config.model_type, f"{model.num_parameters()/1e6:.0f}M params")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

gemma3_text 268M params


## 4. Training configuration

The knobs that decide what the student actually learns: how many epochs, the learning rate, the batch size, the sequence length, the precision, and whether the loss is computed on the whole sequence or only on the assistant turn.

This cell must define `sft_config` (an `SFTConfig`).

In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="gemma3-270m-hop1-sys",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    max_length=256,
    fp16=True,
    assistant_only_loss=True,
    logging_steps=10,
    report_to="none",
)


## 5. Fine-tune

In [8]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,2.607579
20,0.471714
30,0.337646
40,0.364529
50,0.256477
60,0.188595
70,0.142070
80,0.151618
90,0.155571
100,0.103599


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=144, training_loss=0.34971510173959863, metrics={'train_runtime': 92.4782, 'train_samples_per_second': 12.295, 'train_steps_per_second': 1.557, 'total_flos': 70836387508992.0, 'train_loss': 0.34971510173959863, 'entropy': 0.060483720153570175, 'num_tokens': 85239.0, 'mean_token_accuracy': 0.9843940883874893, 'epoch': 3.0})

## 6. Generate Hop-1 decompositions for the test set

Inference has to match training exactly: I apply the same chat template, but stop after the prompt and let the model complete the assistant turn. The gold assistant message is dropped so the model has to produce it.

This cell must define `generate_hop1(messages) -> str`, where `messages` is the full test row's `messages` list (with the gold assistant turn) and the return value is the model's raw YAML string.

In [ ]:
# Define `generate_hop1(messages) -> str`.
def generate_hop1(messages)->str:
  inputs = tokenizer.apply_chat_template(
      messages[:-1],
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt"
  )

  inputs = inputs.to(model.device)

  outputs = model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=False
  )

  input_length = inputs['input_ids'].shape[1]
  generated_tokens = outputs[0][input_length:]

  return tokenizer.decode(generated_tokens, skip_special_tokens=True)


In [10]:
model.eval()

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [ ]:
test_rows = load_dataset(
    'json', data_files='test_sys.jsonl', split='train'
).select_columns(['messages'])

print(test_rows)
print(test_rows[0]['messages'])

In [13]:
print(generate_hop1(test_rows[0]['messages']))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
thought: "I need to find out what kind of product Robinsons is a part of."
action: "Lookup"
target_entity: "Robinsons"


## 7. Run generation over the 95 test questions and save

In [25]:
import json

with open(f"test_{VARIANT}.jsonl") as f:
    test_rows = [json.loads(line) for line in f]

preds = []
for row in test_rows:
    pred_yaml = generate_hop1(row["messages"])
    preds.append({**row, "pred_yaml": pred_yaml})

out_path = f"preds_{VARIANT}.jsonl"
with open(out_path, "w") as f:
    for p in preds:
        f.write(json.dumps(p) + "\n")

print(f"wrote {len(preds)} predictions to {out_path}")
print("--- sample ---")
print(preds[0]["pred_yaml"])

<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tok

In [30]:
# Download the predictions and score them locally through notebook 03.
files.download(out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>